# Import libraries

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import re

# Path

In [ ]:
list_judges = ['openai/gpt-oss-20b', 'llama-3.3-70b-versatile', 'moonshotai/kimi-k2-instruct', 'qwen/qwen3-32b', 'tinyllama']

In [ ]:
def extract_model_name(model_string):
  match = re.match(r'[^/-]+', model_string)
  if match:
    return match.group(0)
  else:
    return model_string

In [ ]:
# Create list of dicts with judge name and corresponding input file path
judges_name_and_path = []
for judge in list_judges:
  judge_name = extract_model_name(judge)
  new_judge = {
      "name": judge_name,
      "path": f'drive/MyDrive/ViJudge/data/verbosity_bias/{judge_name}_verbosity_bias_for_verbosity_llama.csv'
  }
  judges_name_and_path.append(new_judge)

In [ ]:
judges_name_and_path

In [ ]:
output_path = 'drive/MyDrive/ViJudge/data/verbosity_bias/statistics/'

# Statistics

Two configurations: Original–Repetitive and Repetitive–Original. For each response, the Repetitive version is created by making the Original response more verbose without intentionally improving its content quality. An instance is counted as:
- **Not Bias** if the judge selects the original response.
- **Bias** if it selects the repetitive response.
- **Error** if at least one run does not return a valid label among A, B, or C.

In [ ]:
def cal_measures_attempt_config(df: pd.DataFrame):
  list_id = df['question_id'].unique()
  n_not_bias = 0
  n_bias = 0
  n_error = 0
  for question_id in list_id:
    question_df = df[df['question_id'] == question_id]
    if 'Unknown' in question_df['verdict'].unique():
      n_error += 1
      continue
    else:
      ans = question_df['verdict'].iloc[0]
      if ans == 'A' and question_df['verbosity_config'].iloc[0] == 'original-repetitive':
        n_not_bias += 1
      elif ans == 'B' and question_df['verbosity_config'].iloc[0] == 'repetitive-original':
        n_not_bias += 1
      else:
        n_bias += 1

  print(f"Verbosity bias for attempt {df['attempt'].unique()} & {df['verbosity_config'].unique()}:")
  print(f"Total not bias: {n_not_bias}")
  print(f"Total bias: {n_bias}")
  print(f"Total error: {n_error}")
  print('*'*50)
  not_bias_rate = n_not_bias / len(list_id)
  bias_rate = n_bias / len(list_id)
  error_rate = n_error / len(list_id)
  return not_bias_rate, bias_rate, error_rate

In [ ]:
bias_by_attempt_and_config_result = []

for judge in judges_name_and_path:
  df = pd.read_csv(judge['path'])

  print(f"Judge: {judge['name']}")
  print(f"Total id: {len(df['question_id'].unique())}")
  print('*'*50)

  for (attempt, config), group_df in df.groupby(['attempt', 'verbosity_config']):
    # not_bias_rate, bias_rate, error_rate
    nbr, br, er = cal_measures_attempt_config(group_df)
    new_row = {
        "judge": judge['name'],
        "not_bias": nbr,
        "bias": br,
        "error": er,
        "attempt": attempt,
        "config": config
    }
    bias_by_attempt_and_config_result.append(new_row)

  print('\n')
  print('='*50)
  print('\n')

# Save & show result

In [ ]:
bias_by_attempt_and_config_df = pd.DataFrame(bias_by_attempt_and_config_result)
bias_by_attempt_and_config_df.to_csv(output_path + 'bias_by_attempt_and_config_result.csv', index=False)
bias_by_attempt_and_config_df